# Train Inverse Model with Surrogate-Defined Loss (Eigenmode Reconstruction)

**Goal**: Train an inverse MLP that maps eigenmode parameters -> Qiskit Metal design parameters,
using a frozen surrogate model to compute loss in eigenmode space.

**Pipeline**: `eigenmode_params -> inverse_MLP -> qiskit_params -> frozen_surrogate -> reconstructed_eigenmode_params`

Loss = MSE(input_eigenmode, reconstructed_eigenmode) + penalty(qiskit_params outside [0,1])

In [ ]:
## the usual imports
import os, sys, json, csv, gc, warnings
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Dense, Dropout, Input, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import keras_tuner as kt
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## load our hyperparameters from the parameters file
from parameters_surrogate_defined_loss import *

## penalty weight for out-of-range qiskit predictions
PENALTY_WEIGHT = 0.1

## which encoding to use
encoding = 'one_hot'
print(f'Encoding: {encoding}')
print(f'Loss: {TRAIN_LOSS}')
print(f'Penalty weight: {PENALTY_WEIGHT}')
print(f'Keras tuner: {KERAS_TUNER}')



## Load data

In [56]:
## inputs are the hamiltonian values (what we want to invert)
X_train = np.load(f'{DATA_DIR}/npy/x_train_one_hot_encoding_augmented.npy', allow_pickle=True)
X_val   = np.load(f'{DATA_DIR}/npy/x_val_one_hot_encoding_augmented.npy', allow_pickle=True)
X_test  = np.load(f'{DATA_DIR}/npy/x_test_one_hot_encoding_augmented.npy', allow_pickle=True)

## qiskit metal params (all continuous for transmon cross)
y_train = np.load(f'{DATA_DIR}/npy/y_train_one_hot_encoding_augmented.npy', allow_pickle=True)
y_val   = np.load(f'{DATA_DIR}/npy/y_val_one_hot_encoding_augmented.npy', allow_pickle=True)
y_test  = np.load(f'{DATA_DIR}/npy/y_test_one_hot_encoding_augmented.npy', allow_pickle=True)

## column names for printing later
with open('metadata/X_names', 'r') as f:
    eigenmode_column_names = f.read().splitlines()
qiskit_param_names = np.load('metadata/y_columns.npy', allow_pickle=True).astype(str).tolist()

print(f'X_train (eigenmode):  {X_train.shape}')
print(f'y_train (qiskit):     {y_train.shape}')
print(f'X_test  (eigenmode):  {X_test.shape}')
print(f'y_test  (qiskit):     {y_test.shape}')
print(f'Eigenmode columns:    {eigenmode_column_names}')
print(f'Qiskit param columns: {qiskit_param_names}')



X_train (eigenmode):  (567, 2)
y_train (qiskit):     (567, 5)
X_test  (eigenmode):  (122, 2)
y_test  (qiskit):     (122, 5)
Eigenmode columns:    ['cavity_frequency', 'kappa']
Qiskit param columns: ['design_options.claw_opts.connection_pads.readout.claw_length', 'design_options.claw_opts.connection_pads.readout.ground_spacing', 'design_options.cpw_opts.total_length', 'design_options.cpw_opts.meander.asymmetry', 'design_options.cplr_opts.coupling_length']


## Load frozen surrogate model

In [57]:
## load the pre-trained surrogate model
## remember: surrogate maps qiskit params to eigenmode params
## we freeze it so it doesn't get updated during training
surrogate_model_path = 'model/best_keras_model_model2_surrogate.keras'
surrogate_model = load_model(surrogate_model_path)
surrogate_model.trainable = False  ## freeze it babyyyy

print(f'Loaded surrogate from: {surrogate_model_path}')
print(f'Surrogate input shape:  {surrogate_model.input_shape}')
print(f'Surrogate output shape: {surrogate_model.output_shape}')
surrogate_model.summary()



Loaded surrogate from: model/best_keras_model_model2_surrogate.keras
Surrogate input shape:  (None, 5)
Surrogate output shape: (None, 2)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ fc0 (Dense)                     │ (None, 448)            │         2,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu0 (LeakyReLU)         │ (None, 448)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout0 (Dropout)              │ (None, 448)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 2)              │           898 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,760 (42.04 KB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 3,586 (14.01 KB)

 Optimizer params: 7,174 (28.03 KB)

## ScalerConversionLayer

The inverse model's output lives in the inverse scaler space, but the surrogate
expects inputs in ITS scaler space. This layer does the affine transformation
between the two scaler spaces so everything lines up.

In [ ]:
class ScalerConversionLayer(Layer):
    """
    Affine transformation layer to convert between scaler spaces.
    
    The inverse model outputs in one_hot scaler space,
    but the surrogate model expects linear scaler space.
    This layer bridges the gap: linear_scaled = oh_scaled * scale_a + scale_b
    """
    def __init__(self, scale_a, scale_b, **kwargs):
        super().__init__(**kwargs)
        self.scale_a_init = scale_a
        self.scale_b_init = scale_b

    def build(self, input_shape):
        self.scale_a = self.add_weight(
            name='scale_a', shape=self.scale_a_init.shape,
            initializer=tf.keras.initializers.Constant(self.scale_a_init),
            trainable=False)
        self.scale_b = self.add_weight(
            name='scale_b', shape=self.scale_b_init.shape,
            initializer=tf.keras.initializers.Constant(self.scale_b_init),
            trainable=False)
        super().build(input_shape)

    def call(self, inputs):
        return inputs * self.scale_a + self.scale_b

    def get_config(self):
        config = super().get_config()
        config.update({
            'scale_a': self.scale_a_init.tolist() if hasattr(self.scale_a_init, 'tolist') else self.scale_a_init,
            'scale_b': self.scale_b_init.tolist() if hasattr(self.scale_b_init, 'tolist') else self.scale_b_init,
        })
        return config

    @classmethod
    def from_config(cls, config):
        config['scale_a'] = np.array(config['scale_a'])
        config['scale_b'] = np.array(config['scale_b'])
        return cls(**config)

## in the resonator pipeline, qiskit options use a single MinMax scalar space [0, 1],
## meaning no affine transformation is needed between inverse model output and surrogate model input.

n_qiskit_params = y_train.shape[1]
scale_a = np.ones(n_qiskit_params, dtype=np.float32)
scale_b = np.zeros(n_qiskit_params, dtype=np.float32)

for j, col_name in enumerate(qiskit_param_names):
    print(f"{col_name}: scale_a={scale_a[j]:.6f}, scale_b={scale_b[j]:.6f}")

print(f"\nscale_a: {scale_a}")
print(f"scale_b: {scale_b}")

## penalty for predictions that go outside [0, 1] range
## the qiskit metal params should stay in [0, 1] in scaled space
## if they don't, the inverse model is predicting nonsense designs
def qiskit_range_penalty(y_true, y_pred):
    """Penalize predictions outside the valid [0, 1] range."""
    below = tf.nn.relu(-y_pred)       ## how far below 0
    above = tf.nn.relu(y_pred - 1.0)  ## how far above 1
    return tf.reduce_mean(below**2 + above**2)

## the penalty loss needs a dummy y_true (it only looks at y_pred)
dummy_y_train = np.zeros_like(y_train)
dummy_y_val   = np.zeros_like(y_val)
dummy_y_test  = np.zeros_like(y_test)
print('Dummy targets created')



## Build combined model (inverse + frozen surrogate)

In [62]:
## build the combined model:
## eigenmode_input to inverse_MLP to qiskit_params to scaler_conversion to surrogate to reconstructed_eigenmode

n_eigenmode = X_train.shape[1]   ## number of eigenmode params (input to inverse)
n_qiskit    = y_train.shape[1]   ## number of qiskit params (output of inverse)

def build_combined_model(hp=None):
    """Build the combined inverse+surrogate model."""
    
    ## inverse MLP
    inv_input = Input(shape=(n_eigenmode,), name='eigenmode_input')
    x = inv_input
    
    if hp is not None:
        n_layers = hp.Int('n_layers', 3, 6, default=5)
        for i in range(n_layers):
            units = hp.Int(f'units_{i}', 32, 256, step=32, default=64)
            x = Dense(units, activation='relu')(x)
            dr = hp.Float('dropout', 0.0, 0.2, step=0.05, default=0.0)
            if dr > 0:
                x = Dropout(dr)(x)
    else:
        for units in NEURONS_PER_LAYER:
            x = Dense(units, activation='relu')(x)
            if TRAIN_DROPOUT_RATE > 0:
                x = Dropout(TRAIN_DROPOUT_RATE)(x)
    
    inv_output = Dense(n_qiskit, activation='sigmoid', name='qiskit_output')(x)
    inverse_model = Model(inv_input, inv_output, name='inverse_model')
    
    ## scaler conversion + frozen surrogate
    qiskit_converted = ScalerConversionLayer(scale_a, scale_b, name='scaler_conversion')(inv_output)
    eigenmode_reconstructed = surrogate_model(qiskit_converted)
    
    ## combined model
    combined = Model(inputs=inv_input, outputs=[eigenmode_reconstructed, inv_output])
    
    if hp is not None:
        lr = hp.Float('lr', 1e-4, 1e-2, sampling='log', default=LR_INITIAL)
    else:
        lr = LR_INITIAL
    
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=lr,
        decay_steps=len(X_train) // TRAIN_BATCH_SIZE,
        decay_rate=LR_DECAY_RATE,
        staircase=LR_STAIRCASE)
    
    combined.compile(
        optimizer=Adam(learning_rate=lr_schedule),
        loss=[TRAIN_LOSS, qiskit_range_penalty],
        loss_weights=[1.0, PENALTY_WEIGHT],
        metrics={combined.output_names[0]: [TRAIN_LOSS]})
    
    return combined

if not KERAS_TUNER:
    combined_model = build_combined_model()
    combined_model.summary()



## Train the model

In [ ]:
## set up callbacks and paths
best_model_file = f'model/best_keras_model_surrogate_loss.keras'
os.makedirs('model', exist_ok=True)
os.makedirs('plots', exist_ok=True)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=TRAIN_EARLY_STOPPING_PATIENCE,
    restore_best_weights=True,
    verbose=1)

checkpoint = ModelCheckpoint(
    best_model_file,
    monitor='val_loss',
    save_best_only=True,
    verbose=1)

print(f'Best model will be saved to: {best_model_file}')

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    class TrainingPlot(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.losses = []
            self.val_losses = []
        def on_epoch_end(self, epoch, logs=None):
            self.losses.append(logs.get('loss'))
            self.val_losses.append(logs.get('val_loss'))

    class LearningRateMonitor(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.learning_rates = []
        def on_epoch_end(self, epoch, logs=None):
            lr = self.model.optimizer.learning_rate
            if callable(lr):
                lr = lr(self.model.optimizer.iterations)
            self.learning_rates.append(float(tf.keras.backend.get_value(lr)))

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    training_plot = TrainingPlot()
    lr_monitor = LearningRateMonitor()
    
    history = combined_model.fit(
        np.asarray(X_train), [np.asarray(X_train), dummy_y_train],
        epochs=EPOCHS,
        batch_size=TRAIN_BATCH_SIZE,
        validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),
        callbacks=[early_stopping, checkpoint, training_plot, lr_monitor],
        verbose=1)
    
    ## save the final model too just in case
    combined_model.save(best_model_file.replace('.keras', '_final.keras'))
    print(f'Training complete. Best model saved to {best_model_file}')



## Keras Tuner (if enabled)

In [ ]:
if KERAS_TUNER and not SWEEP_PARAM_NUM:
    tuner = kt.BayesianOptimization(
        build_combined_model,
        objective='val_loss',
        max_trials=KERAS_TUNER_TRIALS,
        directory=KERAS_DIR,
        project_name='surrogate_loss_tuner',
        overwrite=False)
    
    tuner.search(
        np.asarray(X_train), [np.asarray(X_train), dummy_y_train],
        epochs=EPOCHS,
        batch_size=TRAIN_BATCH_SIZE,
        validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),
        callbacks=[early_stopping,
                   ModelCheckpoint(best_model_file, monitor='val_loss', save_best_only=True)],
        verbose=1)
    
    tuner.results_summary()

if KERAS_TUNER and not SWEEP_PARAM_NUM and not VISUALIZE_GRADIENTS:
    class LearningRateMonitor(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.learning_rates = []
        def on_epoch_end(self, epoch, logs=None):
            lr = self.model.optimizer.learning_rate
            if callable(lr):
                lr = lr(self.model.optimizer.iterations)
            self.learning_rates.append(float(tf.keras.backend.get_value(lr)))

    best_hp = tuner.get_best_hyperparameters(1)[0]
    model = tuner.hypermodel.build(best_hp)
    lr_monitor = LearningRateMonitor()
    history = model.fit(
        np.asarray(X_train), [np.asarray(X_train), dummy_y_train],
        epochs=400, batch_size=TRAIN_BATCH_SIZE,
        validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),
        callbacks=[early_stopping, lr_monitor], verbose=1)
    del model; tf.keras.backend.clear_session(); gc.collect()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Surrogate-defined loss (total: reconstruction + range penalty)')
plt.ylabel('Loss'); plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='best')
plt.tight_layout(); plt.savefig(f'plots/surrogate_loss_history.pdf'); plt.show()

plt.plot(lr_monitor.learning_rates)
plt.title('Learning Rate over Epochs')
plt.xlabel('Epoch'); plt.ylabel('Learning Rate')
plt.tight_layout(); plt.savefig(f'plots/surrogate_loss_learning_rate.pdf'); plt.show()



### Test set evaluation

In [68]:
## evaluate on test set
tf.keras.backend.clear_session()
gc.collect()

def get_loss(eval_out):
    if isinstance(eval_out, dict):
        return float(eval_out.get("loss", list(eval_out.values())[0]))
    if isinstance(eval_out, (list, tuple, np.ndarray)):
        return float(eval_out[0])
    return float(eval_out)

combined_model = load_model(best_model_file, compile=False,
    custom_objects={"ScalerConversionLayer": ScalerConversionLayer})
combined_model.compile(optimizer="adam",
    loss=[TRAIN_LOSS, qiskit_range_penalty],
    loss_weights=[1.0, PENALTY_WEIGHT])
eval_result = combined_model.evaluate(
    np.asarray(X_test), [np.asarray(X_test), dummy_y_test])
total_loss = float(eval_result[0])
eigenmode_recon_loss = float(eval_result[1])
penalty_loss = float(eval_result[2])

print(f"Total test loss: {total_loss}")
print(f"  Reconstruction loss ({TRAIN_LOSS}): {eigenmode_recon_loss}")
print(f"  Range penalty loss: {penalty_loss} (weighted: {penalty_loss * PENALTY_WEIGHT})")

## check out-of-range using the second output of the combined model directly
predictions = combined_model.predict(np.asarray(X_test), verbose=0)
if isinstance(predictions, list):
    qiskit_pred = predictions[1]
else:
    ## fallback to creating a sub-model pointing to the output if necessary
    qiskit_pred = Model(inputs=combined_model.input, outputs=combined_model.get_layer("qiskit_output").output).predict(np.asarray(X_test), verbose=0)
out_of_range = np.sum((qiskit_pred < 0) | (qiskit_pred > 1))
total_values = qiskit_pred.size
print(f"Out-of-range values: {out_of_range}/{total_values} ({100*out_of_range/total_values:.1f}%)")

test_loss_result = eigenmode_recon_loss



4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 187ms/step - loss: 0.0021 - qiskit_output_loss: 0.0000e+00 - sequential_loss: 0.0020
Total test loss: 0.002052586292847991
  Reconstruction loss (mae): 0.002028496703132987
  Range penalty loss: 0.0 (weighted: 0.0)
Out-of-range values: 0/610 (0.0%)


## Compare predictions vs. test set

In [ ]:
csv_data = [[
    DATA_AUGMENTATION,
    'surrogate_loss_model',
    'InverseModel_SurrogateLoss',
    test_loss_result,
    TRAIN_LOSS,
    TRAIN_BATCH_SIZE,
]]
with open('results/training/test_results.csv', 'a', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(csv_data)
    print(f'Appended to results/training/test_results.csv: {csv_data}')

## predictaroo on test set
tf.keras.backend.clear_session()
gc.collect()

with tf.device("/CPU:0"):
    combined_model = load_model(best_model_file, compile=False,
        custom_objects={"ScalerConversionLayer": ScalerConversionLayer})
    predictions = combined_model.predict(np.asarray(X_test), verbose=0)
    if isinstance(predictions, list):
        eigenmode_reconstructed = predictions[0]
        qiskit_predicted = predictions[1]
    else:
        eigenmode_reconstructed = predictions
        qiskit_predicted = Model(inputs=combined_model.input, outputs=combined_model.get_layer("qiskit_output").output).predict(np.asarray(X_test), verbose=0)

## look at how well the reconstructed eigenmode params match the input eigenmode params

X_test_cur = np.asarray(X_test)
y_test_cur = np.asarray(y_test)  ## ground truth qiskit params (for reference)
eigenmode_recon  = np.asarray(eigenmode_reconstructed)
qiskit_pred = np.asarray(qiskit_predicted)

n_samples, n_eigenmode_cols = X_test_cur.shape
n_qiskit_cols = qiskit_pred.shape[1]
n_samples_to_show = 3

## reconstruction errors (eigenmode space - this is what we trained on)
eigenmode_abs_errors = np.abs(X_test_cur - eigenmode_recon)

print('eigenmode reconstruction for the loss')
for i in range(n_samples_to_show):
    rows = []
    for j in range(n_eigenmode_cols):
        label = eigenmode_column_names[j] if j < len(eigenmode_column_names) else f'eigenmode_col_{j}'
        rows.append({'param': label, 'ref': X_test_cur[i,j],
                     'pred': eigenmode_recon[i,j], 'abs_error': eigenmode_abs_errors[i,j]})
    print(f'- Sample {i} - Eigenmode reconstruction (scaled)')
    print(pd.DataFrame(rows).to_string(index=False))
    
    ## predicted qiskit params
    print(f'\n  Predicted Qiskit Metal params (scaled):')
    for j, col_name in enumerate(qiskit_param_names):
        short = col_name.replace('design_options.', '')
        print(f'    {short:40s}  pred={qiskit_pred[i,j]:.6f}  ref={y_test_cur[i,j]:.6f}  err={abs(qiskit_pred[i,j]-y_test_cur[i,j]):.6f}')
    print()



### Unscaled test vs predictions

In [ ]:
## unscale everything and look at errors in real units that we can actually make sense of
with open("metadata/X_names", "r") as f:
    eigenmode_names = f.read().splitlines()
qiskit_names = np.load("metadata/y_columns.npy", allow_pickle=True).astype(str).tolist()

## unscale input eigenmode params
X_test_unscaled = np.asarray(X_test_cur.copy())
for i in range(X_test_unscaled.shape[0]):
    for j in range(X_test_unscaled.shape[1]):
        eigenmode_name = eigenmode_names[j] if j < len(eigenmode_names) else f"col_{j}"
        scaler = joblib.load(f"scalers/scaler_X_linear_{eigenmode_name}.save")
        X_test_unscaled[i, j] = scaler.inverse_transform([[X_test_unscaled[i, j]]])[0][0]

## unscale reconstructed eigenmode params
eigenmode_recon_unscaled = np.asarray(eigenmode_recon.copy())
for i in range(eigenmode_recon_unscaled.shape[0]):
    for j in range(eigenmode_recon_unscaled.shape[1]):
        eigenmode_name = eigenmode_names[j] if j < len(eigenmode_names) else f"col_{j}"
        scaler = joblib.load(f"scalers/scaler_X_linear_{eigenmode_name}.save")
        eigenmode_recon_unscaled[i, j] = scaler.inverse_transform([[eigenmode_recon_unscaled[i, j]]])[0][0]

## unscale qiskit param predictions
qiskit_pred_unscaled = np.asarray(qiskit_pred.copy())
y_test_unscaled = np.asarray(y_test_cur.copy())
for i in range(qiskit_pred_unscaled.shape[0]):
    for j in range(qiskit_pred_unscaled.shape[1]):
        col_name = qiskit_names[j] if j < len(qiskit_names) else f"col_{j}"
        scaler = joblib.load(f"scalers/scaler_y_linear_{col_name}.save")
        qiskit_pred_unscaled[i, j] = scaler.inverse_transform([[qiskit_pred_unscaled[i, j]]])[0][0]
        y_test_unscaled[i, j] = scaler.inverse_transform([[y_test_unscaled[i, j]]])[0][0]

n_samples_to_show = 3
eigenmode_abs_unscaled = np.abs(X_test_unscaled - eigenmode_recon_unscaled)

data = {}
for name in eigenmode_names:
    data[f"ref_{name}"] = []
    data[f"pred_{name}"] = []
for name in qiskit_names:
    short = name.replace("design_options.", "")
    data[f"ref_{short}"] = []
    data[f"pred_{short}"] = []

print('unscaled eigenmode reconstruction')
for i in range(n_samples_to_show):
    rows = []
    for j in range(X_test_unscaled.shape[1]):
        rows.append({"param": eigenmode_names[j], "ref_unscaled": X_test_unscaled[i,j],
                     "pred_unscaled": eigenmode_recon_unscaled[i,j], "abs_error": eigenmode_abs_unscaled[i,j]})
        data[f"ref_{eigenmode_names[j]}"] += [X_test_unscaled[i,j]]
        data[f"pred_{eigenmode_names[j]}"] += [eigenmode_recon_unscaled[i,j]]
    print(f"- Sample {i} (Unscaled) - Eigenmode reconstruction")
    
    print(pd.DataFrame(rows).to_string(index=False))
    
    print(f"\n  Predicted Qiskit Metal params (unscaled):")
    for j, col_name in enumerate(qiskit_names):
        short = col_name.replace("design_options.", "")
        print(f"    {short:40s}  pred={qiskit_pred_unscaled[i,j]:.6f}  ref={y_test_unscaled[i,j]:.6f}  err={abs(qiskit_pred_unscaled[i,j]-y_test_unscaled[i,j]):.6f}")
        data[f"pred_{short}"] += [qiskit_pred_unscaled[i,j]]
        data[f"ref_{short}"] += [y_test_unscaled[i,j]]

    print(f"\n  Reference Qiskit Metal params (unscaled):")
    for j, col_name in enumerate(qiskit_names):
        short = col_name.replace("design_options.", "")
        print(f"    {short:40s}  ref={y_test_unscaled[i,j]:.6f}")
    print()

print("Unscaled eigenmode reconstruction error stats:")
print("  min:", float(eigenmode_abs_unscaled.min()),
      " median:", float(np.median(eigenmode_abs_unscaled)),
      " max:", float(eigenmode_abs_unscaled.max()))

pd.DataFrame(data).to_csv("results/validation/eigenmode-based-full_MLP_results.csv",index = False)

